# 03 · Energy vs cable length: a two-objective layout problem (NSGA-II)

Spreading turbines out reduces wake losses but needs more inter-array cable, and cable is where most
of a wind farm's copper sits. This notebook searches for the whole trade-off rather than a single answer.

- **Objective 1:** maximise AEP (full Horns Rev 1 wind climate, same model as notebooks 01–02)
- **Objective 2:** minimise inter-array cable length, estimated as the minimum spanning tree (MST)
  connecting all turbines
- **Constraint:** every pair of turbines at least 4 D apart, inside the same 2 km × 1 km lease area as notebook 02

The MST is a lower bound on real cable length: it ignores cable capacity limits (how many turbines
one string can carry), the substation position and seabed routing.

In [ ]:
%pip install -q py_wake==2.6.20 topfarm==2.6.2 pymoo==0.6.2

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree

from py_wake.examples.data.hornsrev1 import Hornsrev1Site, V80
from py_wake.literature.gaussian_models import Bastankhah_PorteAgel_2014
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize

FIG = Path("figures"); FIG.mkdir(exist_ok=True)

site, wt = Hornsrev1Site(), V80()
wf_model = Bastankhah_PorteAgel_2014(site, wt, k=0.0324555)

LEASE_X, LEASE_Y = 2000, 1000
MIN_SPACING = 4 * wt.diameter()
POP_SIZE, N_GEN, SEED = 40, 100, 1

x_init = np.linspace(0, 1600, 4).repeat(2)
y_init = np.tile(np.linspace(0, 400, 2), 4)
n_wt = len(x_init)

In [ ]:
def mst_edges(x, y):
    """Minimum spanning tree over the turbines, as a sparse matrix of edge lengths (m)."""
    return minimum_spanning_tree(squareform(pdist(np.column_stack([x, y]))))

def cable_length_km(x, y):
    return mst_edges(x, y).sum() / 1000


class LayoutProblem(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=2 * n_wt, n_obj=2, n_ieq_constr=1,
                         xl=[0] * (2 * n_wt),
                         xu=[LEASE_X] * n_wt + [LEASE_Y] * n_wt)

    def _evaluate(self, v, out, *args, **kwargs):
        x, y = v[:n_wt], v[n_wt:]
        aep = wf_model(x, y).aep().sum().values
        out["F"] = [-aep, cable_length_km(x, y)]        # pymoo minimises, so AEP is negated
        # Feasible when the closest pair is at least MIN_SPACING apart (G <= 0)
        out["G"] = [MIN_SPACING - pdist(np.column_stack([x, y])).min()]


def initial_population(problem, seed_row):
    # Random layouts, with the starting grid as the first member so the search
    # always has one feasible, sensible layout to build from.
    rng = np.random.default_rng(SEED)
    X0 = rng.uniform(problem.xl, problem.xu, size=(POP_SIZE, problem.n_var))
    X0[0] = seed_row
    return X0


problem = LayoutProblem()
X0 = initial_population(problem, np.concatenate([x_init, y_init]))
result = minimize(problem, NSGA2(pop_size=POP_SIZE, sampling=X0), ("n_gen", N_GEN), seed=SEED, verbose=False)
print(f"{len(result.F)} non-dominated layouts found")

In [ ]:
aep_init = wf_model(x_init, y_init).aep().sum().values
cable_init = cable_length_km(x_init, y_init)

pareto = pd.DataFrame({
    "AEP_GWh": -result.F[:, 0],
    "Cable_km": result.F[:, 1],
}).assign(GWh_per_km_cable=lambda d: d.AEP_GWh / d.Cable_km) \
  .sort_values("AEP_GWh").reset_index(drop=True)

print(pareto.round(2).to_string())
print(f"\nInitial 4x2 grid: {aep_init:.2f} GWh, {cable_init:.2f} km, "
      f"{aep_init / cable_init:.2f} GWh/km")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(pareto.Cable_km, pareto.AEP_GWh, label="Pareto front (NSGA-II)")
plt.scatter(cable_init, aep_init, color="red", marker="x", s=100, label="Initial 4x2 grid")
plt.xlabel("Inter-array cable length, MST (km)")
plt.ylabel("AEP (GWh)")
plt.title("Energy vs cable length, 8 × V80 at Horns Rev 1")
plt.legend(); plt.grid(True, alpha=0.4)
plt.savefig(FIG / "03_pareto_front.png", dpi=150, bbox_inches="tight")
plt.show()

## What the two ends of the front look like

In [ ]:
ends = {"Shortest cable": np.argmin(result.F[:, 1]), "Highest AEP": np.argmin(result.F[:, 0])}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (label, idx) in zip(axes, ends.items()):
    x, y = result.X[idx, :n_wt], result.X[idx, n_wt:]
    for i, j in zip(*mst_edges(x, y).nonzero()):
        ax.plot([x[i], x[j]], [y[i], y[j]], color="darkorange", lw=2)
    ax.scatter(x, y, s=80, zorder=3)
    ax.plot([0, LEASE_X, LEASE_X, 0, 0], [0, 0, LEASE_Y, LEASE_Y, 0], "k--", alpha=0.5)
    ax.set_title(f"{label}: {-result.F[idx, 0]:.1f} GWh, {result.F[idx, 1]:.2f} km")
    ax.set_xlabel("x (m)"); ax.set_aspect("equal"); ax.grid(True, alpha=0.3)
axes[0].set_ylabel("y (m)")
plt.savefig(FIG / "03_front_extremes.png", dpi=150, bbox_inches="tight")
plt.show()